In [ ]:
import inspect
from aion.modalities import LegacySurveyImage
from aion.codecs import CodecManager

print(inspect.getsource(LegacySurveyImage))

In [ ]:
import json, os, csv
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import pearsonr
from aion import AION
from aion.modalities import LegacySurveyImage

device = "cuda" if torch.cuda.is_available() else "mps"   # MPS support unverified
model = AION.from_pretrained("polymathic-ai/aion-base").to(device).eval()
codec_manager = CodecManager(device=device)

with open("cutouts_split/split_manifest.json") as fh:
    manifest = json.load(fh)
train_files = [f"cutouts_split/train/{n}" for n in manifest["train"]]
test_files  = [f"cutouts_split/test/{n}"  for n in manifest["test"]]

In [ ]:
g, r, z = np.asarray(np.load(train_files[0], allow_pickle=True).item()["flux"], dtype=np.float32)
flux3 = torch.from_numpy(np.stack([g, r, z])).unsqueeze(0).to(device)

img = LegacySurveyImage(flux=flux3, bands=["DES-G", "DES-R", "DES-Z"])
emb = model.encode(codec_manager.encode(img), num_encoder_tokens=600)
print("3-band accepted — shape:", emb.shape)

In [ ]:
def emb_for(flux, bands):
    img = LegacySurveyImage(flux=flux, bands=bands)
    with torch.no_grad():
        return model.encode(codec_manager.encode(img), num_encoder_tokens=600).cpu().numpy()

a = emb_for(flux3, ["DES-G", "DES-R", "DES-Z"])
b = emb_for(flux3, ["DES-G", "DES-R", "DES-I"]) 
c = emb_for(flux3[:, [2, 1, 0]], ["DES-Z", "DES-R", "DES-G"])  

print("label swap  Z->I  :", np.abs(a - b).max())
print("channel reorder    :", np.abs(a - c).max())


In [ ]:
import json, os, csv
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import pearsonr

from aion import AION
from aion.codecs import CodecManager
from aion.modalities import LegacySurveyImage

TORCH_SEED = 42
BANDS = ["DES-G", "DES-R", "DES-Z"]
BATCH = 32

device = "cuda" if torch.cuda.is_available() else "cpu"
model = AION.from_pretrained("polymathic-ai/aion-base").to(device).eval()
codec_manager = CodecManager(device=device)

with open("cutouts_split/split_manifest.json") as fh:
    manifest = json.load(fh)
train_files = [f"cutouts_split/train/{n}" for n in manifest["train"]]
test_files  = [f"cutouts_split/test/{n}"  for n in manifest["test"]]
print(len(train_files), len(test_files))

In [ ]:
def load_flux(p):
    g, r, z = np.asarray(np.load(p, allow_pickle=True).item()["flux"], dtype=np.float32)
    return np.stack([g, r, z])

def embed_aion(paths):
    out = []
    for i in range(0, len(paths), BATCH):
        arr = np.stack([load_flux(p) for p in paths[i:i+BATCH]])
        flux = torch.from_numpy(arr).float().to(device)
        img = LegacySurveyImage(flux=flux, bands=BANDS)
        with torch.no_grad():
            emb = model.encode(codec_manager.encode(img), num_encoder_tokens=600)
        out.append(emb.mean(dim=1).cpu().numpy())
        print(f"{min(i+BATCH, len(paths))}/{len(paths)}", flush=True)
    return np.concatenate(out)

if os.path.exists("train_embeddings_aion.npy"):
    train_embeddings = np.load("train_embeddings_aion.npy")
    test_embeddings  = np.load("test_embeddings_aion.npy")
else:
    train_embeddings = embed_aion(train_files)
    np.save("train_embeddings_aion.npy", train_embeddings)
    test_embeddings = embed_aion(test_files)
    np.save("test_embeddings_aion.npy", test_embeddings)

print(train_embeddings.shape, test_embeddings.shape)

In [10]:
meta = pd.read_parquet("sample_2k_matches_metadata.parquet").set_index("_healpix_29")

train_ids = [int(os.path.splitext(n)[0]) for n in manifest["train"]]
test_ids  = [int(os.path.splitext(n)[0]) for n in manifest["test"]]

train_targets = meta.loc[train_ids, "LOG_MSTAR_mmu_desi_provabgs"].values.astype(np.float32)
test_targets  = meta.loc[test_ids,  "LOG_MSTAR_mmu_desi_provabgs"].values.astype(np.float32)

target_mean, target_std = train_targets.mean(), train_targets.std()
train_targets_norm = (train_targets - target_mean) / target_std

print(train_targets.shape, test_targets.shape, "nans:", np.isnan(train_targets).sum())

(1600,) (400,) nans: 0


In [ ]:
n_val = int(len(train_embeddings) * 0.15)
X_train_t = torch.from_numpy(train_embeddings[n_val:]).float()
X_val_t   = torch.from_numpy(train_embeddings[:n_val]).float()
y_train_t = torch.from_numpy(train_targets_norm[n_val:]).float().unsqueeze(1)
y_val_t   = torch.from_numpy(train_targets_norm[:n_val]).float().unsqueeze(1)

embed_dim = train_embeddings.shape[1]

torch.manual_seed(TORCH_SEED)
probe = nn.Linear(embed_dim, 1)
optimizer = torch.optim.Adam(probe.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

out_dir = f"aion_probe_results/seed_{TORCH_SEED}"
os.makedirs(out_dir, exist_ok=True)

best_val_loss, patience, patience_counter, max_epochs = float("inf"), 20, 0, 200

with open(f"{out_dir}/training_log.csv", "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["epoch", "train_loss", "val_loss"])
    for epoch in range(max_epochs):
        probe.train()
        optimizer.zero_grad()
        train_loss = loss_fn(probe(X_train_t), y_train_t)
        train_loss.backward()
        optimizer.step()

        probe.eval()
        with torch.no_grad():
            val_loss = loss_fn(probe(X_val_t), y_val_t)

        w.writerow([epoch, train_loss.item(), val_loss.item()])

        if val_loss.item() < best_val_loss:
            best_val_loss, patience_counter = val_loss.item(), 0
            torch.save(probe.state_dict(), "probe_best.pt")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"early stopping at epoch {epoch}")
                break

print(f"best val loss: {best_val_loss:.4f}")

best val loss: 0.4101


In [ ]:
probe_eval = nn.Linear(embed_dim, 1)
probe_eval.load_state_dict(torch.load("probe_best.pt"))
probe_eval.eval()

with torch.no_grad():
    pred_norm = probe_eval(torch.from_numpy(test_embeddings).float()).squeeze(1).numpy()
pred = pred_norm * target_std + target_mean

summary = {
    "mae":  float(np.mean(np.abs(pred - test_targets))),
    "rmse": float(np.sqrt(np.mean((pred - test_targets) ** 2))),
    "r2":   float(1 - np.sum((pred - test_targets) ** 2) / np.sum((test_targets - test_targets.mean()) ** 2)),
    "pearson_r": float(pearsonr(pred, test_targets)[0]),
    "n_test": len(test_targets),
}
with open(f"{out_dir}/eval_summary_aion.json", "w") as fh:
    json.dump(summary, fh, indent=2)
print(summary)

{'mae': 0.29441726207733154, 'rmse': 0.40794309973716736, 'r2': 0.5889168977737427, 'pearson_r': 0.767827570438385, 'n_test': 400}


In [14]:
SEEDS = [2, 4, 9, 16, 21, 42, 100, 216, 331, 369, 402, 4021]

n_val = int(len(train_embeddings) * 0.15)
X_train_t = torch.from_numpy(train_embeddings[n_val:]).float()
X_val_t   = torch.from_numpy(train_embeddings[:n_val]).float()
y_train_t = torch.from_numpy(train_targets_norm[n_val:]).float().unsqueeze(1)
y_val_t   = torch.from_numpy(train_targets_norm[:n_val]).float().unsqueeze(1)
X_test_t  = torch.from_numpy(test_embeddings).float()
embed_dim = train_embeddings.shape[1]
loss_fn = nn.MSELoss()

for seed in SEEDS:
    out_dir = f"aion_probe_results/seed_{seed}"
    os.makedirs(out_dir, exist_ok=True)
    if os.path.exists(f"{out_dir}/eval_summary_aion.json"):
        print(f"seed {seed} — done, skipping")
        continue

    torch.manual_seed(seed)
    probe = nn.Linear(embed_dim, 1)
    optimizer = torch.optim.Adam(probe.parameters(), lr=1e-3)

    best_val_loss, patience_counter = float("inf"), 0
    with open(f"{out_dir}/training_log.csv", "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["epoch", "train_loss", "val_loss"])
        for epoch in range(200):
            probe.train()
            optimizer.zero_grad()
            tl = loss_fn(probe(X_train_t), y_train_t)
            tl.backward()
            optimizer.step()

            probe.eval()
            with torch.no_grad():
                vl = loss_fn(probe(X_val_t), y_val_t)
            w.writerow([epoch, tl.item(), vl.item()])

            if vl.item() < best_val_loss:
                best_val_loss, patience_counter = vl.item(), 0
                torch.save(probe.state_dict(), "probe_best.pt")
            else:
                patience_counter += 1
                if patience_counter >= 20:
                    break

    probe_eval = nn.Linear(embed_dim, 1)
    probe_eval.load_state_dict(torch.load("probe_best.pt"))
    probe_eval.eval()
    with torch.no_grad():
        pred = probe_eval(X_test_t).squeeze(1).numpy() * target_std + target_mean

    summary = {
        "mae":  float(np.mean(np.abs(pred - test_targets))),
        "rmse": float(np.sqrt(np.mean((pred - test_targets) ** 2))),
        "r2":   float(1 - np.sum((pred - test_targets) ** 2) / np.sum((test_targets - test_targets.mean()) ** 2)),
        "pearson_r": float(pearsonr(pred, test_targets)[0]),
        "n_test": len(test_targets),
    }
    with open(f"{out_dir}/eval_summary_aion.json", "w") as fh:
        json.dump(summary, fh, indent=2)
    print(f"seed {seed}: {summary}")

seed 2: {'mae': 0.3035711348056793, 'rmse': 0.42122969031333923, 'r2': 0.561703085899353, 'pearson_r': 0.7501668930053711, 'n_test': 400}
seed 4: {'mae': 0.29797470569610596, 'rmse': 0.4130910634994507, 'r2': 0.5784763097763062, 'pearson_r': 0.7612426280975342, 'n_test': 400}
seed 9: {'mae': 0.29473230242729187, 'rmse': 0.40961095690727234, 'r2': 0.5855486392974854, 'pearson_r': 0.7656481266021729, 'n_test': 400}
seed 16: {'mae': 0.29504746198654175, 'rmse': 0.4106706976890564, 'r2': 0.5834013223648071, 'pearson_r': 0.7642841339111328, 'n_test': 400}
seed 21: {'mae': 0.3038237690925598, 'rmse': 0.42046454548835754, 'r2': 0.563293993473053, 'pearson_r': 0.7512555122375488, 'n_test': 400}
seed 42 — done, skipping
seed 100: {'mae': 0.3043076694011688, 'rmse': 0.42268261313438416, 'r2': 0.5586743354797363, 'pearson_r': 0.7484718561172485, 'n_test': 400}
seed 216: {'mae': 0.30583834648132324, 'rmse': 0.4239348769187927, 'r2': 0.55605548620224, 'pearson_r': 0.7470630407333374, 'n_test': 400}

| Metric | Mean ± SD |
|:------|:---------:|
| **MAE ↓** | **0.299 ± 0.004** |
| **RMSE ↓** | **0.416 ± 0.006** |
| **R² ↑** | **0.573 ± 0.011** |
| **Pearson r ↑** | **0.758 ± 0.007** |

